# Expert-CFG: VQA-RAD 5-Run Ablation (논문 Table 1 재현)

| Run | 설명 | γ | LoRA | RAG | Attn | Embed | Selective |
|-----|------|---|------|-----|------|-------|-----------|
| 1. baseline | Phi3.5V-Med (no RAG) | 1.0 | ✅ | ✗ | ✗ | ✗ | ✗ |
| 2. w/ RAG | 전체 샘플 RAG | 1.0 | ✅ | ✅ | ✗ | ✗ | ✗ |
| 3. w/ 5% RAG | 상위 5% PE 샘플만 RAG | 1.0 | ✅ | ✅ | ✗ | ✗ | ✅ 5% |
| 4. w/ Expert-RAG | 5% + attn+embed (γ=1.0) | 1.0 | ✅ | ✅ | ✅ | ✅ | ✅ 5% |
| 5. w/ 5% Expert-CFG | 5% + full CFG (γ=1.5) ← **논문 메인** | 1.5 | ✅ | ✅ | ✅ | ✅ | ✅ 5% |

**논문 Table 1 목표치 (VQA-RAD):**
- baseline: Open=69.27%, Closed=90.44%, Overall=82.04%
- 5% Expert-CFG: Open=74.86%, Closed=91.18%, Overall=84.70%

**필요한 Kaggle 데이터셋:**
```
/kaggle/input/expert-cfg-data/
├── test.json          # annotation + top_k_captions + highlights
└── VQA-RAD/images/

/kaggle/input/datasets/ciderchicken/expert-cfg-weights/
├── Phi-3.5-vision-instruct/   # 베이스 모델 가중치
└── lora_weights/
    └── logs_phi35_pubmed_instruct/   # Phi3.5V-Med LoRA
```

## Step 1. 패키지 설치

In [ ]:
!pip install -q \
    "transformers==4.48.3" \
    "accelerate==0.30.0" \
    "peft==0.11.1" \
    bitsandbytes

import importlib
import transformers, tokenizers
importlib.reload(transformers)
print("transformers:", transformers.__version__)   # 4.48.3
print("tokenizers  :", tokenizers.__version__)     # 0.21.x

## Step 2. Expert-CFG 클론

In [ ]:
import os, subprocess, sys, json, shutil

REPO_DIR = '/kaggle/working/Expert-CFG'

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run([
    'git', 'clone', '--branch', 'feat/vqa-rad',
    'https://github.com/minseongkim-dev/expert-cfg',
    REPO_DIR,
], check=True)
print('클론 완료')

## Step 3. 모델 가중치 복사

레포 안의 `Phi-3.5-vision-instruct/`에는 커스텀 processor(`processing_phi3_v_cfg.py`)가 있음.  
Kaggle 데이터셋의 가중치(.safetensors 등)를 그 폴더로 복사해서 합침.

In [ ]:
import shutil

WEIGHTS_SRC = '/kaggle/input/datasets/ciderchicken/expert-cfg-weights/Phi-3.5-vision-instruct'
MODEL_DIR   = '/kaggle/working/Expert-CFG/Phi-3.5-vision-instruct'

copied = 0
for fname in os.listdir(WEIGHTS_SRC):
    src = os.path.join(WEIGHTS_SRC, fname)
    dst = os.path.join(MODEL_DIR, fname)
    if os.path.isfile(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f'복사: {fname}')
        copied += 1

print(f'완료: {copied}개 파일 복사')

In [ ]:
# 호환성 패치
import json, shutil

# --- HF 모듈 캐시 초기화 ---
modules_cache = os.path.expanduser('~/.cache/huggingface/modules/transformers_modules')
if os.path.exists(modules_cache):
    shutil.rmtree(modules_cache)
    print('✅ transformers_modules 캐시 초기화')

# --- tokenizer_config.json AutoProcessor 교정 ---
# transformers가 tokenizer_config.json을 processor_config.json 이후에 읽어 덮어쓰므로 수정
tok_cfg_path = os.path.join(MODEL_DIR, 'tokenizer_config.json')
if os.path.exists(tok_cfg_path):
    with open(tok_cfg_path) as f:
        tok_cfg = json.load(f)
    old = tok_cfg.get('auto_map', {}).get('AutoProcessor', '')
    if old != 'processing_phi3_v_cfg.Phi3VProcessor':
        tok_cfg.setdefault('auto_map', {})['AutoProcessor'] = 'processing_phi3_v_cfg.Phi3VProcessor'
        with open(tok_cfg_path, 'w') as f:
            json.dump(tok_cfg, f, indent=2)
        print(f'✅ tokenizer_config.json AutoProcessor 교정: {old!r} → processing_phi3_v_cfg.Phi3VProcessor')
    else:
        print('ℹ️  tokenizer_config.json 이미 올바름')
else:
    print('⚠️  tokenizer_config.json 없음 — 건너뜀')

## Step 4. 경로 확인

In [ ]:
DATA_ROOT = '/kaggle/input/expert-cfg-data'
TEST_JSON = f'{DATA_ROOT}/test.json'
IMG_ROOT  = f'{DATA_ROOT}/VQA-RAD/images'
WORK_DIR  = '/kaggle/working'
LORA_DIR  = '/kaggle/input/datasets/ciderchicken/expert-cfg-weights/lora_weights/logs_phi35_pubmed_instruct'

for label, path in [
    ('model', MODEL_DIR),
    ('test.json', TEST_JSON),
    ('images/', IMG_ROOT),
    ('lora weights', LORA_DIR),
]:
    print(f"{('✅' if os.path.exists(path) else '❌')} {label}: {path}")

if os.path.exists(IMG_ROOT):
    print(f'   이미지 수: {len([f for f in os.listdir(IMG_ROOT) if f.endswith(".jpg")])}개')

## Step 5. 5가지 실행 (논문 Table 1 순서)

> **실행 순서 중요**: Run 1 → Run 2 → (Run 3/4/5는 Run 1 결과가 필요).
> 각 셀 약 1-2시간 소요.

In [ ]:
# Run 1: Phi3.5V-Med baseline (LoRA, no RAG, γ=1.0)
# → 논문 Table 1 첫 번째 행: Open=69.27%, Closed=90.44%, Overall=82.04% 목표
cmd = [
    sys.executable, 'demo.py',
    '--dataset',            'vqa-rad',
    '--input_json',         TEST_JSON,
    '--img_root',           IMG_ROOT,
    '--model_name_or_path', MODEL_DIR,
    '--use_qlora',
    '--output_dir',         LORA_DIR,
    '--bf16',
    '--num_crops',          '4',
    '--cfg',                '1.0',
    '--no_rag',
    '--save_path',          f'{WORK_DIR}/result_1_baseline.json',
]
print('▶ Run 1: Phi3.5V-Med baseline (no RAG, γ=1.0, LoRA)')
subprocess.run(cmd, cwd='/kaggle/working/Expert-CFG')

In [ ]:
# Run 2: w/ RAG — 전체 샘플 RAG, γ=1.0 (CFG 없음)
cmd = [
    sys.executable, 'demo.py',
    '--dataset',            'vqa-rad',
    '--input_json',         TEST_JSON,
    '--img_root',           IMG_ROOT,
    '--model_name_or_path', MODEL_DIR,
    '--use_qlora',
    '--output_dir',         LORA_DIR,
    '--bf16',
    '--num_crops',          '4',
    '--cfg',                '1.0',
    '--save_path',          f'{WORK_DIR}/result_2_rag_all.json',
]
print('▶ Run 2: w/ RAG (all samples, γ=1.0, LoRA)')
subprocess.run(cmd, cwd='/kaggle/working/Expert-CFG')

In [ ]:
# Run 3: w/ 5% RAG — 상위 5% PE 샘플만 RAG 적용 (selective_rerun.py)
# Phase 1 결과(result_1_baseline.json)에서 PE 상위 5% 선별 → RAG만 적용 (γ=1.0)
cmd = [
    sys.executable, '../selective_rerun.py',
    '--baseline',           f'{WORK_DIR}/result_1_baseline.json',
    '--top_k_pct',          '5.0',
    '--dataset',            'vqa-rad',
    '--input_json',         TEST_JSON,
    '--img_root',           IMG_ROOT,
    '--model_name_or_path', MODEL_DIR,
    '--use_qlora',
    '--output_dir',         LORA_DIR,
    '--bf16',
    '--num_crops',          '4',
    '--cfg',                '1.0',
    '--save_path',          f'{WORK_DIR}/result_3_5pct_rag.json',
]
print('▶ Run 3: w/ 5% RAG (selective, γ=1.0, LoRA)')
subprocess.run(cmd, cwd='/kaggle/working/Expert-CFG')

In [ ]:
# Run 4: w/ Expert-RAG — 5% + attn patch + embedding perturb, γ=1.0 (CFG bias 없음)
cmd = [
    sys.executable, '../selective_rerun.py',
    '--baseline',           f'{WORK_DIR}/result_1_baseline.json',
    '--top_k_pct',          '5.0',
    '--dataset',            'vqa-rad',
    '--input_json',         TEST_JSON,
    '--img_root',           IMG_ROOT,
    '--model_name_or_path', MODEL_DIR,
    '--use_qlora',
    '--output_dir',         LORA_DIR,
    '--bf16',
    '--num_crops',          '4',
    '--cfg',                '1.0',
    '--attn',               '3.0',
    '--perturb_weight',     '0.01',
    '--use_attn_patch',
    '--use_embed_perturb',
    '--save_path',          f'{WORK_DIR}/result_4_expert_rag.json',
]
print('▶ Run 4: w/ Expert-RAG (5% selective, attn+embed, γ=1.0, LoRA)')
subprocess.run(cmd, cwd='/kaggle/working/Expert-CFG')

In [ ]:
# Run 5: w/ 5% Expert-CFG — 논문 메인 결과 (γ=1.5, attn+embed, 5% selective)
# 목표: Open=74.86%, Closed=91.18%, Overall=84.70%
cmd = [
    sys.executable, '../selective_rerun.py',
    '--baseline',           f'{WORK_DIR}/result_1_baseline.json',
    '--top_k_pct',          '5.0',
    '--dataset',            'vqa-rad',
    '--input_json',         TEST_JSON,
    '--img_root',           IMG_ROOT,
    '--model_name_or_path', MODEL_DIR,
    '--use_qlora',
    '--output_dir',         LORA_DIR,
    '--bf16',
    '--num_crops',          '4',
    '--cfg',                '1.5',
    '--attn',               '3.0',
    '--perturb_weight',     '0.01',
    '--use_attn_patch',
    '--use_embed_perturb',
    '--save_path',          f'{WORK_DIR}/result_5_5pct_expert_cfg.json',
]
print('▶ Run 5: w/ 5% Expert-CFG (γ=1.5, attn+embed, 5% selective, LoRA) ← 논문 메인')
subprocess.run(cmd, cwd='/kaggle/working/Expert-CFG')

## Step 6. 결과 비교

In [ ]:
# ── 논문 Table 1 목표치 ──────────────────────────────────────────────────────
paper = {
    '1. baseline':     (90.44, 69.27, 82.04),
    '5. w/ 5% ExpCFG': (91.18, 74.86, 84.70),
}

runs = [
    ('1. baseline',       f'{WORK_DIR}/result_1_baseline.json',       '1.0', 'N', 'N'),
    ('2. w/ RAG',         f'{WORK_DIR}/result_2_rag_all.json',         '1.0', 'N', 'N'),
    ('3. w/ 5% RAG',      f'{WORK_DIR}/result_3_5pct_rag.json',        '1.0', 'N', '5%'),
    ('4. w/ Expert-RAG',  f'{WORK_DIR}/result_4_expert_rag.json',      '1.0', 'Y', '5%'),
    ('5. w/ 5% ExpCFG',   f'{WORK_DIR}/result_5_5pct_expert_cfg.json', '1.5', 'Y', '5%'),
]

print(f"{'Run':<22} {'Closed':>8} {'Open':>7} {'Overall':>8}  γ   Attn  Sel   nC   nO")
print('─' * 82)

for name, path, cfg, attn, sel in runs:
    if not os.path.exists(path):
        print(f"{name:<22}  (미완료)")
        continue
    with open(path) as f:
        data = json.load(f)
    s = data['summary']
    c_acc = s.get('closed_accuracy', float('nan')) * 100
    o_acc = s.get('open_accuracy',   float('nan')) * 100
    ov    = s['accuracy'] * 100
    nc    = s.get('closed_total', 0)
    no_   = s.get('open_total',   0)

    paper_mark = ''
    if name in paper:
        pc, po, pov = paper[name]
        dc, do_, dov = c_acc - pc, o_acc - po, ov - pov
        paper_mark = f'  ← paper C={pc:.2f}({dc:+.2f}) O={po:.2f}({do_:+.2f}) OA={pov:.2f}({dov:+.2f})'

    print(f"{name:<22} {c_acc:>7.2f}%  {o_acc:>6.2f}%  {ov:>7.2f}%  {cfg}   {attn}   {sel:>3}  {nc:>3}  {no_:>3}{paper_mark}")